In [17]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import (
    Adam,
    SGD,
    RMSprop
)
import keras_tuner as kt
from tensorflow.keras.applications import (
    MobileNetV2,
    EfficientNetB0,
    ResNet50,
    DenseNet121
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D
)
import keras_tuner as kt
import pandas as pd

In [2]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

In [3]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
train_df["label"] = encoder.fit_transform(train_df["disease"])
valid_df["label"] = encoder.transform(valid_df["disease"])
test_df["label"] = encoder.transform(test_df["disease"])

In [4]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 32

In [5]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [6]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    return image, label

In [7]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [8]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [9]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [10]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(0.21338772031292808), 5: np.float64(1.285530900421786), 6: np.float64(10.115440115440116)}


In [11]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [12]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [15]:
base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False

In [18]:
mobilenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [19]:
mobilenet.compile(
    optimizer="SGD",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [20]:
history_mobile = mobilenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 104s 451ms/step - accuracy: 0.3867 - loss: 1.6442 - val_accuracy: 0.2157 - val_loss: 1.9760 - learning_rate: 0.0100
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 139s 434ms/step - accuracy: 0.5167 - loss: 1.3321 - val_accuracy: 0.3036 - val_loss: 2.0561 - learning_rate: 0.0100
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 100s 445ms/step - accuracy: 0.5479 - loss: 1.2279 - val_accuracy: 0.4900 - val_loss: 1.2938 - learning_rate: 0.0100
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 96s 425ms/step - accuracy: 0.5586 - loss: 1.1718 - val_accuracy: 0.6218 - val_loss: 1.0458 - learning_rate: 0.0100
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 104s 462ms/step - accuracy: 0.5817 - loss: 1.1241 - val_accuracy: 0.5499 - val_loss: 1.2285 - learning_rate: 0.0100
Restoring model weights from the end of the best epoch: 4.


In [21]:
train_loss, lr_train_acc_mob = mobilenet.evaluate(train_ds)
valid_loss, lr_valid_acc_mob = mobilenet.evaluate(valid_ds)
test_loss, lr_test_acc_mob = mobilenet.evaluate(test_ds)
print(lr_train_acc_mob)
print(lr_valid_acc_mob)
print(lr_test_acc_mob)

220/220 ━━━━━━━━━━━━━━━━━━━━ 71s 310ms/step - accuracy: 0.6204 - loss: 1.0205
47/47 ━━━━━━━━━━━━━━━━━━━━ 20s 421ms/step - accuracy: 0.6079 - loss: 1.0508
47/47 ━━━━━━━━━━━━━━━━━━━━ 17s 359ms/step - accuracy: 0.6048 - loss: 1.0613
0.6203994154930115
0.6078562140464783
0.6047903895378113


In [22]:
mob_results = pd.DataFrame(columns=[
    "Model",
    "Train accuracy",
    "Test accuracy",
    "Valid accuracy"
])
mob_results.loc[len(mob_results)] = [
    "mobilenet using SGD",
    lr_train_acc_mob,
    lr_test_acc_mob,
    lr_valid_acc_mob
]
mob_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,mobilenet using SGD,0.620399,0.60479,0.607856


In [23]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [24]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [25]:
base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False

In [26]:
mobilenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [27]:
mobilenet.compile(
    optimizer="RMSprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [28]:
history_mobile = mobilenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 105s 458ms/step - accuracy: 0.4568 - loss: 1.6460 - val_accuracy: 0.4494 - val_loss: 1.3763 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 104s 462ms/step - accuracy: 0.5395 - loss: 1.3351 - val_accuracy: 0.6471 - val_loss: 0.9482 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 138s 441ms/step - accuracy: 0.5685 - loss: 1.2089 - val_accuracy: 0.6052 - val_loss: 1.0541 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 96s 427ms/step - accuracy: 0.5757 - loss: 1.1717 - val_accuracy: 0.5526 - val_loss: 1.1613 - learning_rate: 0.0010
Epoch 5/5
219/220 ━━━━━━━━━━━━━━━━━━━━ 0s 365ms/step - accuracy: 0.5982 - loss: 1.1326
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
220/220 ━━━━━━━━━━━━━━━━━━━━ 99s 441ms/step - accuracy: 0.5981 - loss: 1.1329 - val_accuracy: 0.4714 - val_loss: 1.3757 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 2.


In [29]:
train_loss, rm_train_acc_mob = mobilenet.evaluate(train_ds)
valid_loss, rm_valid_acc_mob = mobilenet.evaluate(valid_ds)
test_loss, rm_test_acc_mob = mobilenet.evaluate(test_ds)
print(rm_train_acc_mob)
print(rm_valid_acc_mob)
print(rm_test_acc_mob)

220/220 ━━━━━━━━━━━━━━━━━━━━ 82s 359ms/step - accuracy: 0.6685 - loss: 0.9100
47/47 ━━━━━━━━━━━━━━━━━━━━ 17s 350ms/step - accuracy: 0.6664 - loss: 0.9305
47/47 ━━━━━━━━━━━━━━━━━━━━ 16s 344ms/step - accuracy: 0.6287 - loss: 1.0008
0.6684736013412476
0.666444718837738
0.628742516040802


In [30]:
mob_results.loc[len(mob_results)] = [
    "mobilenet using RMSprop",
    rm_train_acc_mob,
    rm_test_acc_mob,
    rm_valid_acc_mob
]
mob_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,mobilenet using SGD,0.620399,0.604790,0.607856
1,mobilenet using RMSprop,0.668474,0.628743,0.666445


In [31]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [32]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [33]:
base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False

In [34]:
mobilenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [35]:
mobilenet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [36]:
history_mobile = mobilenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 105s 454ms/step - accuracy: 0.4631 - loss: 1.5811 - val_accuracy: 0.5393 - val_loss: 1.2641 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 111s 494ms/step - accuracy: 0.5613 - loss: 1.2325 - val_accuracy: 0.6678 - val_loss: 0.9648 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 69s 300ms/step - accuracy: 0.5675 - loss: 1.1556 - val_accuracy: 0.4980 - val_loss: 1.4210 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 355ms/step - accuracy: 0.5833 - loss: 1.0883 - val_accuracy: 0.6079 - val_loss: 1.0592 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 130s 579ms/step - accuracy: 0.5927 - loss: 1.0635 - val_accuracy: 0.6511 - val_loss: 0.9464 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 5.


In [37]:
train_loss, adam_train_acc_mob = mobilenet.evaluate(train_ds)
valid_loss, adam_valid_acc_mob = mobilenet.evaluate(valid_ds)
test_loss, adam_test_acc_mob = mobilenet.evaluate(test_ds)
print(adam_train_acc_mob)
print(adam_valid_acc_mob)
print(adam_test_acc_mob)

220/220 ━━━━━━━━━━━━━━━━━━━━ 103s 456ms/step - accuracy: 0.6797 - loss: 0.8824
47/47 ━━━━━━━━━━━━━━━━━━━━ 21s 440ms/step - accuracy: 0.6551 - loss: 0.9407
47/47 ━━━━━━━━━━━━━━━━━━━━ 23s 496ms/step - accuracy: 0.6474 - loss: 0.9675
0.679743230342865
0.6551265120506287
0.6473719477653503


In [38]:
mob_results.loc[len(mob_results)] = [
    "mobilenet using adam",
    adam_train_acc_mob,
    adam_test_acc_mob,
    adam_valid_acc_mob
]
mob_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,mobilenet using SGD,0.620399,0.604790,0.607856
1,mobilenet using RMSprop,0.668474,0.628743,0.666445
2,mobilenet using adam,0.679743,0.647372,0.655127


In [39]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 16

In [40]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [41]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    return image, label

In [42]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [43]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [44]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [45]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(0.21338772031292808), 5: np.float64(1.285530900421786), 6: np.float64(10.115440115440116)}


In [46]:
base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False

In [47]:
mobilenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [48]:
mobilenet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [49]:
batch_size = 16
history_mobile = mobilenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    batch_size = batch_size
)

Epoch 1/5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


439/439 ━━━━━━━━━━━━━━━━━━━━ 145s 319ms/step - accuracy: 0.4498 - loss: 1.5310 - val_accuracy: 0.5752 - val_loss: 1.1244
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 158s 355ms/step - accuracy: 0.5459 - loss: 1.2363 - val_accuracy: 0.5759 - val_loss: 1.1810
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 142s 317ms/step - accuracy: 0.5612 - loss: 1.1748 - val_accuracy: 0.6045 - val_loss: 1.0982
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 143s 319ms/step - accuracy: 0.5805 - loss: 1.1069 - val_accuracy: 0.5180 - val_loss: 1.2159
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 122s 272ms/step - accuracy: 0.5940 - loss: 1.0379 - val_accuracy: 0.5959 - val_loss: 1.0385


In [50]:
train_loss, bt16_train_acc_mob = mobilenet.evaluate(train_ds)
valid_loss, bt16_valid_acc_mob = mobilenet.evaluate(valid_ds)
test_loss, bt16_test_acc_mob = mobilenet.evaluate(test_ds)
print(bt16_train_acc_mob)
print(bt16_valid_acc_mob)
print(bt16_test_acc_mob)

439/439 ━━━━━━━━━━━━━━━━━━━━ 87s 195ms/step - accuracy: 0.6093 - loss: 0.9888
94/94 ━━━━━━━━━━━━━━━━━━━━ 19s 206ms/step - accuracy: 0.5885 - loss: 1.0501
94/94 ━━━━━━━━━━━━━━━━━━━━ 20s 209ms/step - accuracy: 0.5995 - loss: 1.0749
0.6092724800109863
0.5885486006736755
0.5994677543640137


In [51]:
mob_results.loc[len(mob_results)] = [
    "mobilenet using batchsize  16",
    bt16_train_acc_mob,
    bt16_test_acc_mob,
    bt16_valid_acc_mob
]
mob_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,mobilenet using SGD,0.620399,0.604790,0.607856
1,mobilenet using RMSprop,0.668474,0.628743,0.666445
2,mobilenet using adam,0.679743,0.647372,0.655127
3,mobilenet using batchsize 16,0.609272,0.599468,0.588549


In [52]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [53]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [55]:
base_model = mobilenet.layers[0]

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_mobile_ft = mobilenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 144s 313ms/step - accuracy: 0.5108 - loss: 1.4527 - val_accuracy: 0.5593 - val_loss: 1.1777 - learning_rate: 1.0000e-05
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 140s 313ms/step - accuracy: 0.5200 - loss: 1.1648 - val_accuracy: 0.5553 - val_loss: 1.2291 - learning_rate: 1.0000e-05
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 144s 323ms/step - accuracy: 0.5397 - loss: 1.0995 - val_accuracy: 0.5293 - val_loss: 1.2880 - learning_rate: 1.0000e-05
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 270ms/step - accuracy: 0.5629 - loss: 1.0304
Epoch 4: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
439/439 ━━━━━━━━━━━━━━━━━━━━ 140s 313ms/step - accuracy: 0.5629 - loss: 1.0304 - val_accuracy: 0.5246 - val_loss: 1.2632 - learning_rate: 1.0000e-05
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 147s 328ms/step - accuracy: 0.5802 - loss: 0.9589 - val_accuracy: 0.5579 - val_loss: 1.1865 - learning_rate: 5.0000e-06
Restoring model weights from the end of the best epo

In [56]:
train_loss, fine_train_acc_mob = mobilenet.evaluate(train_ds)
valid_loss, fine_valid_acc_mob = mobilenet.evaluate(valid_ds)
test_loss, fine_test_acc_mob = mobilenet.evaluate(test_ds)
print(fine_train_acc_mob)
print(fine_valid_acc_mob)
print(fine_test_acc_mob)

439/439 ━━━━━━━━━━━━━━━━━━━━ 73s 160ms/step - accuracy: 0.5649 - loss: 1.1252
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 95ms/step - accuracy: 0.5426 - loss: 1.1852
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 97ms/step - accuracy: 0.5403 - loss: 1.2060
0.5649072527885437
0.5426098704338074
0.5402528047561646


In [57]:
mob_results.loc[len(mob_results)] = [
    "mobilenet fine tunning",
    fine_train_acc_mob,
    fine_test_acc_mob,
    fine_valid_acc_mob
]
mob_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,mobilenet using SGD,0.620399,0.604790,0.607856
1,mobilenet using RMSprop,0.668474,0.628743,0.666445
2,mobilenet using adam,0.679743,0.647372,0.655127
3,mobilenet using batchsize 16,0.609272,0.599468,0.588549
4,mobilenet fine tunning,0.564907,0.540253,0.542610


In [58]:
def build_mobilenet(hp):
    base = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=INPUT_SHAPE
    )
    base.trainable = True
    model = tf.keras.Sequential([
        base,
        GlobalAveragePooling2D(),
        Dense(
            hp.Int(
                "units",
                min_value=128,
                max_value=512,
                step=128
            ),
            activation="relu"
        ),
        Dropout(
            hp.Float(
                "dropout",
                min_value=0.2,
                max_value=0.5,
                step=0.1
            )
        ),
        Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=hp.Float(
                "learning_rate",
                min_value=1e-5,
                max_value=1e-3,
                sampling="log"
            )
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [59]:
mobilenet_tuner = kt.RandomSearch(
    build_mobilenet,
    objective="val_accuracy",
    max_trials=3,
    directory="tuning",
    project_name="mobilenet"
)
mobilenet_tuner.search(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights
)

Trial 3 Complete [00h 19m 07s]
val_accuracy: 0.606524646282196

Best val_accuracy So Far: 0.6704394221305847
Total elapsed time: 00h 57m 57s


In [60]:
best_mobilenet = mobilenet_tuner.get_best_models(1)[0]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 322 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [61]:
best_hps = mobilenet_tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'units': 128, 'dropout': 0.2, 'learning_rate': 0.00013812005903276895}


In [62]:
base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False

In [63]:
mobilenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [64]:
mobilenet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [65]:
history_mobile = mobilenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5
)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 52s 111ms/step - accuracy: 0.7026 - loss: 0.8592 - val_accuracy: 0.7324 - val_loss: 0.7456
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 54s 120ms/step - accuracy: 0.7374 - loss: 0.7299 - val_accuracy: 0.7164 - val_loss: 0.7866
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 52s 115ms/step - accuracy: 0.7449 - loss: 0.7044 - val_accuracy: 0.7430 - val_loss: 0.6953
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 50s 111ms/step - accuracy: 0.7581 - loss: 0.6742 - val_accuracy: 0.7523 - val_loss: 0.6847
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 50s 111ms/step - accuracy: 0.7619 - loss: 0.6519 - val_accuracy: 0.7437 - val_loss: 0.7343


In [66]:
train_loss, hype_train_acc_mob = mobilenet.evaluate(train_ds)
valid_loss, hype_valid_acc_mob = mobilenet.evaluate(valid_ds)
test_loss, hype_test_acc_mob = mobilenet.evaluate(test_ds)
print(hype_train_acc_mob)
print(hype_valid_acc_mob)
print(hype_test_acc_mob)

439/439 ━━━━━━━━━━━━━━━━━━━━ 41s 90ms/step - accuracy: 0.7558 - loss: 0.6513
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 93ms/step - accuracy: 0.7430 - loss: 0.7265
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 95ms/step - accuracy: 0.7259 - loss: 0.7841
0.7557774782180786
0.7430093288421631
0.7258815765380859


In [67]:
mob_results.loc[len(mob_results)] = [
    "mobilenet using hyperparameter",
    hype_train_acc_mob,
    hype_test_acc_mob,
    hype_valid_acc_mob
]
mob_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,mobilenet using SGD,0.620399,0.604790,0.607856
1,mobilenet using RMSprop,0.668474,0.628743,0.666445
2,mobilenet using adam,0.679743,0.647372,0.655127
3,mobilenet using batchsize 16,0.609272,0.599468,0.588549
4,mobilenet fine tunning,0.564907,0.540253,0.542610
5,mobilenet using hyperparameter,0.755777,0.725882,0.743009


In [68]:
mob_results.to_csv("mobilenet_comparison.csv",index=False)

In [69]:
best_mobilenet.save("cnn_mobilenet_phase5.keras")

In [70]:
mobilenet.save("cnn_mobilenet_bestmodel.keras")